# 01 Data Workspace

Goal:

1. inspect the raw format of the three datasets
2. initialize the `dataset/<SCENE>/part1/` workspace for each scene
3. extract unified `shared/raw_images/` and `shared/meta/`
4. resize to `shared/images_512/`
5. generate unified subset files

This notebook only handles dataset workspace preparation. It does not run COLMAP or VGGT reconstruction.


In [11]:
from pathlib import Path
import json
import shutil
from PIL import Image

CV_ROOT = Path('/home/bzhang512/CV_Project')
DATASET_ROOT = CV_ROOT / 'dataset'

SCENES = ['Re10k-1', 'DL3DV-2', '405841']
TARGET_SIZE = (512, 512)
SUBSET_SIZES = [96]

#SCENE = 'Re10k-1'   # 修改这里切换数据集
#SCENE = 'DL3DV-2'
SCENE = '405841'
OVERWRITE = False
LINK_INSTEAD_OF_COPY = True

SCENE_ROOT = DATASET_ROOT / SCENE
PART1_ROOT = SCENE_ROOT / 'part1'
SHARED_ROOT = PART1_ROOT / 'shared'
RAW_IMAGES_DIR = SHARED_ROOT / 'raw_images'
IMAGES_512_DIR = SHARED_ROOT / 'images_512'
META_DIR = SHARED_ROOT / 'meta'
SUBSETS_DIR = SHARED_ROOT / 'subsets'
PLAN_A_DIR = PART1_ROOT / 'planA'
PLAN_B_DIR = PART1_ROOT / 'planB'

print('SCENE_ROOT =', SCENE_ROOT)
print('PART1_ROOT =', PART1_ROOT)


SCENE_ROOT = /home/bzhang512/CV_Project/dataset/405841
PART1_ROOT = /home/bzhang512/CV_Project/dataset/405841/part1


## 1. Scene format adapter

This section normalizes different raw dataset layouts and identifies:
- image directory
- metadata source
- scene type


In [12]:
def get_scene_spec(scene_name: str):
    root = DATASET_ROOT / scene_name
    if scene_name == 'Re10k-1':
        return {
            'scene_type': 're10k',
            'image_dir': root / 'images',
            'cameras_json': root / 'cameras.json',
            'intrinsics_json': root / 'intrinsics.json',
        }
    if scene_name == 'DL3DV-2':
        return {
            'scene_type': 'dl3dv',
            'image_dir': root / 'rgb',
            'cameras_json': root / 'cameras.json',
            'intrinsics_json': root / 'intrinsics.json',
        }
    if scene_name == '405841':
        return {
            'scene_type': 'front_rgb_calib',
            'image_dir': root / 'FRONT' / 'rgb',
            'calib_dir': root / 'FRONT' / 'calib',
            'depth_dir': root / 'FRONT' / 'depth',
            'gt_dir': root / 'FRONT' / 'gt',
        }
    raise ValueError(f'Unsupported scene: {scene_name}')

spec = get_scene_spec(SCENE)
spec


{'scene_type': 'front_rgb_calib',
 'image_dir': PosixPath('/home/bzhang512/CV_Project/dataset/405841/FRONT/rgb'),
 'calib_dir': PosixPath('/home/bzhang512/CV_Project/dataset/405841/FRONT/calib'),
 'depth_dir': PosixPath('/home/bzhang512/CV_Project/dataset/405841/FRONT/depth'),
 'gt_dir': PosixPath('/home/bzhang512/CV_Project/dataset/405841/FRONT/gt')}

## 2. Check dataset format


In [13]:
image_dir = spec['image_dir']
print('image_dir exists:', image_dir.exists(), image_dir)

image_files = sorted([p for p in image_dir.iterdir() if p.suffix.lower() in {'.png', '.jpg', '.jpeg'}])
print('num images =', len(image_files))

if image_files:
    with Image.open(image_files[0]) as im:
        print('first image size =', im.size)
        print('first image =', image_files[0].name)

for k, v in spec.items():
    if k != 'image_dir':
        print(k, '=>', v, '| exists =', Path(v).exists())


image_dir exists: True /home/bzhang512/CV_Project/dataset/405841/FRONT/rgb
num images = 199
first image size = (1920, 1280)
first image = 000000.png
scene_type => front_rgb_calib | exists = False
calib_dir => /home/bzhang512/CV_Project/dataset/405841/FRONT/calib | exists = True
depth_dir => /home/bzhang512/CV_Project/dataset/405841/FRONT/depth | exists = True
gt_dir => /home/bzhang512/CV_Project/dataset/405841/FRONT/gt | exists = True


## 3. Initialize per-scene workspace


In [14]:
for d in [PART1_ROOT, SHARED_ROOT, RAW_IMAGES_DIR, IMAGES_512_DIR, META_DIR, SUBSETS_DIR, PLAN_A_DIR, PLAN_B_DIR]:
    d.mkdir(parents=True, exist_ok=True)
    print('ready:', d)


ready: /home/bzhang512/CV_Project/dataset/405841/part1
ready: /home/bzhang512/CV_Project/dataset/405841/part1/shared
ready: /home/bzhang512/CV_Project/dataset/405841/part1/shared/raw_images
ready: /home/bzhang512/CV_Project/dataset/405841/part1/shared/images_512
ready: /home/bzhang512/CV_Project/dataset/405841/part1/shared/meta
ready: /home/bzhang512/CV_Project/dataset/405841/part1/shared/subsets
ready: /home/bzhang512/CV_Project/dataset/405841/part1/planA
ready: /home/bzhang512/CV_Project/dataset/405841/part1/planB


## 4. Extract raw_images into the shared workspace


In [16]:
def safe_copy_or_link(src: Path, dst: Path, link=False, overwrite=False):
    if dst.exists() or dst.is_symlink():
        if not overwrite:
            return
        if dst.is_symlink() or dst.is_file():
            dst.unlink()
        elif dst.is_dir():
            shutil.rmtree(dst)
    if link:
        dst.symlink_to(src)
    else:
        shutil.copy2(src, dst)

for src in image_files:
    dst = RAW_IMAGES_DIR / src.name
    safe_copy_or_link(src, dst, link=LINK_INSTEAD_OF_COPY, overwrite=OVERWRITE)

print('raw_images ready:', RAW_IMAGES_DIR)
print('count =', len(list(RAW_IMAGES_DIR.iterdir())))


raw_images ready: /home/bzhang512/CV_Project/dataset/405841/part1/shared/raw_images
count = 199


## 5. Write metadata


In [17]:
ordered_images = sorted([p.name for p in RAW_IMAGES_DIR.iterdir() if p.is_file()])
(META_DIR / 'image_list.txt').write_text('\n'.join(ordered_images) + '\n')


summary = {
    'scene': SCENE,
    'scene_type': spec['scene_type'],
    'image_dir': str(spec['image_dir']),
    'num_images': len(ordered_images),
    'target_size': list(TARGET_SIZE),
}

for key in ['cameras_json', 'intrinsics_json', 'calib_dir', 'depth_dir', 'gt_dir']:
    if key in spec:
        summary[key] = str(spec[key])

(META_DIR / 'dataset_info.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False))

if 'cameras_json' in spec and Path(spec['cameras_json']).exists():
    shutil.copy2(spec['cameras_json'], META_DIR / 'cameras.json')
if 'intrinsics_json' in spec and Path(spec['intrinsics_json']).exists():
    shutil.copy2(spec['intrinsics_json'], META_DIR / 'intrinsics.json')

if 'calib_dir' in spec and Path(spec['calib_dir']).exists():
    calib_files = sorted(Path(spec['calib_dir']).glob('*.txt'))
    calib_summary = {
        'num_calib_files': len(calib_files),
        'first_calib_file': calib_files[0].name if calib_files else None,
    }
    (META_DIR / 'calib_summary.json').write_text(json.dumps(calib_summary, indent=2, ensure_ascii=False))

print('meta written to', META_DIR)


meta written to /home/bzhang512/CV_Project/dataset/405841/part1/shared/meta


## 6. Resize to 512


In [18]:
raw_files = sorted([p for p in RAW_IMAGES_DIR.iterdir() if p.suffix.lower() in {'.png', '.jpg', '.jpeg'}])
for src in raw_files:
    dst = IMAGES_512_DIR / src.name
    if dst.exists() and not OVERWRITE:
        continue
    with Image.open(src) as im:
        resized = im.resize(TARGET_SIZE, Image.LANCZOS)
        resized.save(dst)

print('images_512 ready:', IMAGES_512_DIR)
print('count =', len(list(IMAGES_512_DIR.iterdir())))


images_512 ready: /home/bzhang512/CV_Project/dataset/405841/part1/shared/images_512
count = 199


## 7. Generate subset


In [19]:
def uniform_subset(names, k):
    if k >= len(names):
        return names
    if k <= 1:
        return [names[0]]
    idxs = [round(i * (len(names) - 1) / (k - 1)) for i in range(k)]
    return [names[i] for i in idxs]

names_512 = sorted([p.name for p in IMAGES_512_DIR.iterdir() if p.is_file()])
for k in SUBSET_SIZES:
    subset = uniform_subset(names_512, k)
    out = SUBSETS_DIR / f"subset_{k}.txt"
    out.write_text("\n".join(subset) + "\n", encoding="utf-8")
    print("wrote", out, "count =", len(subset))


wrote /home/bzhang512/CV_Project/dataset/405841/part1/shared/subsets/subset_96.txt count = 96


## 8. Build `scene/images` from a subset txt

This utility is shared by later Plan A / Plan B notebooks. The subset txt remains the canonical definition, while this cell provides a unified way to materialize a `scene/images/` directory.


In [20]:
def build_scene_images_from_subset(subset_name: str, scene_dir: Path, link=False, overwrite=False):
    subset_file = SUBSETS_DIR / subset_name
    names = [line.strip() for line in subset_file.read_text().splitlines() if line.strip()]
    images_dir = scene_dir / 'images'
    images_dir.mkdir(parents=True, exist_ok=True)

    for name in names:
        src = IMAGES_512_DIR / name
        dst = images_dir / name
        if dst.exists() or dst.is_symlink():
            if not overwrite:
                continue
            if dst.is_symlink() or dst.is_file():
                dst.unlink()
        if link:
            dst.symlink_to(src)
        else:
            shutil.copy2(src, dst)

    print('scene/images ready:', images_dir)
    print('count =', len(list(images_dir.iterdir())))
    return images_dir

# example:
# build_scene_images_from_subset('subset_96.txt', PART1_ROOT / '_tmp_subset96_scene', link=False, overwrite=False)


## 9. Final check


In [ ]:
for d in [RAW_IMAGES_DIR, IMAGES_512_DIR, META_DIR, SUBSETS_DIR, PLAN_A_DIR, PLAN_B_DIR]:
    print("\n", d)
    items = sorted(d.iterdir())
    for p in items[:10]:
        print(" ", p.name)
    if len(items) > 10:
        print(" ...")



 /home/bzhang512/CV_Project/dataset/405841/part1/shared/raw_images
  000000.png
  000001.png
  000002.png
  000003.png
  000004.png
  000005.png
  000006.png
  000007.png
  000008.png
  000009.png
 ...

 /home/bzhang512/CV_Project/dataset/405841/part1/shared/images_512
  000000.png
  000001.png
  000002.png
  000003.png
  000004.png
  000005.png
  000006.png
  000007.png
  000008.png
  000009.png
 ...

 /home/bzhang512/CV_Project/dataset/405841/part1/shared/meta
  calib_summary.json
  dataset_info.json
  image_list.txt

 /home/bzhang512/CV_Project/dataset/405841/part1/shared/subsets
  subset_96.txt

 /home/bzhang512/CV_Project/dataset/405841/part1/planA

 /home/bzhang512/CV_Project/dataset/405841/part1/planB


: 